Scenario: Automated Business Report Assistant
🎯 Problem Statement
In companies:
- Managers spend hours compiling data from multiple sources (CRM, ERP, spreadsheets).
- Reports often require manual formatting, chart creation, and consistency checks.
- This slows down decision-making and introduces human error.
👉 Solution
A multi-agent AutoGen system where:
- 🤖 Assistant Agent drafts report templates and queries relevant data.
- ⚙️ UserProxy Agent executes data pulls and formatting automatically.
- 🔁 The system iterates until the report meets formatting, accuracy, and completeness standards.
🔄 Workflow
Business Request → AI drafts report structure → Data fetched & formatted → Charts auto-generated → Validation loop → Final polished report

In [1]:
pip install pandas matplotlib openai

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import random

# ==========================================
# MOCK DATA SOURCE (Simulating CRM / ERP)
# ==========================================
def fetch_sales_data():
    data = {
        "Month": ["Jan", "Feb", "Mar", "Apr", "May"],
        "Revenue": [random.randint(1000, 5000) for _ in range(5)],
        "Expenses": [random.randint(500, 3000) for _ in range(5)]
    }
    df = pd.DataFrame(data)
    df["Profit"] = df["Revenue"] - df["Expenses"]
    return df


# ==========================================
# 🤖 Assistant Agent (Draft Report Structure)
# ==========================================
class AssistantAgent:
    def draft_report(self, business_request):
        print("\n🤖 Assistant Agent: Drafting report structure...\n")

        report = {
            "title": "Monthly Business Report",
            "sections": [
                "Executive Summary",
                "Revenue Analysis",
                "Expense Analysis",
                "Profit Insights",
                "Charts"
            ],
            "status": "draft"
        }
        return report


# ==========================================
# ⚙️ UserProxy Agent (Executes Tasks)
# ==========================================
class UserProxyAgent:

    def get_data(self):
        print("⚙️ UserProxy Agent: Fetching data...\n")
        return fetch_sales_data()

    def generate_chart(self, df):
        print("⚙️ Generating charts...\n")

        plt.figure()
        plt.plot(df["Month"], df["Revenue"], marker='o', label="Revenue")
        plt.plot(df["Month"], df["Expenses"], marker='o', label="Expenses")
        plt.plot(df["Month"], df["Profit"], marker='o', label="Profit")

        plt.title("Business Performance")
        plt.legend()
        plt.savefig("report_chart.png")
        plt.close()

        return "report_chart.png"


# ==========================================
# 🔁 Validator Agent (Quality Check)
# ==========================================
class ValidatorAgent:

    def validate(self, report, df):
        print("🔁 Validating report...\n")

        if df.isnull().sum().sum() > 0:
            return False, "Missing data found"

        if len(report["sections"]) < 5:
            return False, "Incomplete report sections"

        return True, "Report is valid"


# ==========================================
# 📊 Report Generator
# ==========================================
def generate_final_report(report, df, chart_path):
    print("📊 Generating final report...\n")

    summary = f"""
    {report['title']}
    ============================

    Executive Summary:
    Total Revenue: {df['Revenue'].sum()}
    Total Expenses: {df['Expenses'].sum()}
    Total Profit: {df['Profit'].sum()}

    Chart saved at: {chart_path}
    """

    return summary


# ==========================================
# 🔄 MAIN WORKFLOW
# ==========================================
def run_business_report_system():

    business_request = "Generate monthly business performance report"

    assistant = AssistantAgent()
    user_proxy = UserProxyAgent()
    validator = ValidatorAgent()

    # Step 1: Draft Report
    report = assistant.draft_report(business_request)

    while True:
        # Step 2: Fetch Data
        df = user_proxy.get_data()

        # Step 3: Generate Chart
        chart_path = user_proxy.generate_chart(df)

        # Step 4: Validate
        is_valid, message = validator.validate(report, df)
        print("Validation:", message)

        if is_valid:
            break
        else:
            print("🔁 Fixing issues and retrying...\n")

    # Step 5: Final Report
    final_report = generate_final_report(report, df, chart_path)

    print("\n✅ FINAL REPORT\n")
    print(final_report)


# ==========================================
# RUN SYSTEM
# ==========================================
if __name__ == "__main__":
    run_business_report_system()


🤖 Assistant Agent: Drafting report structure...

⚙️ UserProxy Agent: Fetching data...

⚙️ Generating charts...

🔁 Validating report...

Validation: Report is valid
📊 Generating final report...


✅ FINAL REPORT


    Monthly Business Report
    
    Executive Summary:
    Total Revenue: 17369
    Total Expenses: 10747
    Total Profit: 6622
    
    Chart saved at: report_chart.png
    


In [12]:
!pip install groq pandas matplotlib

In [1]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [5]:
!pip install groq
import os
import pandas as pd
import matplotlib.pyplot as plt
import random
from groq import Groq

# Initialize Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# ==========================================
# 📊 MOCK DATA SOURCE
# ==========================================
def fetch_sales_data():
    data = {
        "Month": ["Jan", "Feb", "Mar", "Apr", "May"],
        "Revenue": [random.randint(2000, 6000) for _ in range(5)],
        "Expenses": [random.randint(1000, 4000) for _ in range(5)]
    }
    df = pd.DataFrame(data)
    df["Profit"] = df["Revenue"] - df["Expenses"]
    return df


# ==========================================
# 🤖 Assistant Agent (Groq LLM)
# ==========================================
class AssistantAgent:

    def generate_report(self, df):
        print("🤖 Assistant Agent (Groq): Generating report...\n")

        prompt = f"""
        You are a business analyst AI.

        Analyze the following data and generate a structured report:

        {df.to_string()}

        Include:
        - Executive Summary
        - Key Insights
        - Trends
        - Recommendations
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )

        return response.choices[0].message.content


# ==========================================
# ⚙️ UserProxy Agent
# ==========================================
class UserProxyAgent:

    def get_data(self):
        print("⚙️ Fetching data...\n")
        return fetch_sales_data()

    def generate_chart(self, df):
        print("⚙️ Generating chart...\n")

        plt.figure()
        plt.plot(df["Month"], df["Revenue"], marker='o', label="Revenue")
        plt.plot(df["Month"], df["Expenses"], marker='o', label="Expenses")
        plt.plot(df["Month"], df["Profit"], marker='o', label="Profit")

        plt.title("Business Performance")
        plt.legend()

        chart_path = "report_chart.png"
        plt.savefig(chart_path)
        plt.close()

        return chart_path


# ==========================================
# 🔁 Validator Agent (Groq LLM)
# ==========================================
class ValidatorAgent:

    def validate(self, report):
        print("🔁 Validator Agent: Validating report...\n")

        prompt = f"""
        Validate the following report:

        {report}

        Check:
        - Completeness
        - Clarity
        - Missing insights

        Reply ONLY:
        VALID or INVALID + reason
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        result = response.choices[0].message.content

        if "VALID" in result:
            return True, result
        return False, result


# ==========================================
# 🔄 MAIN WORKFLOW
# ==========================================
def run_system():

    assistant = AssistantAgent()
    user_proxy = UserProxyAgent()
    validator = ValidatorAgent()

    # Step 1: Get Data
    df = user_proxy.get_data()

    while True:
        # Step 2: Generate Report
        report = assistant.generate_report(df)

        # Step 3: Generate Chart
        chart_path = user_proxy.generate_chart(df)

        # Step 4: Validate
        is_valid, feedback = validator.validate(report)
        print("Validation:", feedback, "\n")

        if is_valid:
            break
        else:
            print("🔁 Improving report...\n")

    # Step 5: Final Output
    print("\n✅ FINAL REPORT\n")
    print(report)
    print("\n📊 Chart saved at:", chart_path)


# ==========================================
# ▶️ RUN
# ==========================================
run_system()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 9.1 MB/s eta 0:00:00
⚙️ Fetching data...

🤖 Assistant Agent (Groq): Generating report...

⚙️ Generating chart...

🔁 Validator Agent: Validating report...

Validation: VALID

Reason: 

- The report is comprehensive, covering key insights, trends, and recommendations.
- The language is clear and concise, making it easy to understand the analysis and recommendations.
- The report provides a clear executive summary, key insights, trends, and recommendations, making it easy to follow.
- The report includes a detailed analysis of revenue, expense, and profit trends, as well as recommendations for growth, expense management, and profit maximization.
- The report also includes next steps and a conclusion, providing a clear direction for the business.
- The report does not miss any major insights, such as:
  - A comparison of the company's performance to industry benchmarks or competitors.
  - An analysis of the impact of external facto

In [6]:
import os
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

models = client.models.list()

print("Available Groq Models:")
for model in models.data:
    print(f"- {model.id}")

Available Groq Models:
- meta-llama/llama-prompt-guard-2-86m
- openai/gpt-oss-120b
- allam-2-7b
- openai/gpt-oss-safeguard-20b
- canopylabs/orpheus-v1-english
- groq/compound-mini
- llama-3.1-8b-instant
- moonshotai/kimi-k2-instruct-0905
- moonshotai/kimi-k2-instruct
- whisper-large-v3
- openai/gpt-oss-20b
- llama-3.3-70b-versatile
- meta-llama/llama-4-scout-17b-16e-instruct
- canopylabs/orpheus-arabic-saudi
- groq/compound
- meta-llama/llama-prompt-guard-2-22m
- whisper-large-v3-turbo
- qwen/qwen3-32b


Scenario: AI Marketing Content & Validation System (Corporate Marketing Team)
🎯 Problem Statement
In companies:
- Marketing teams draft campaign content manually.
- Editors check for tone, compliance, and brand alignment.
- Testing across platforms (emails, social posts, ads) is manual and time-consuming.
👉 Solution
A multi-agent system where:
- ✍️ Content Creator Agent generates marketing copy, visuals, or ad text.
- 🔎 Validator Agent reviews for compliance, tone, and brand guidelines, then simulates deployment.
- 🔁 The loop continues until the content meets quality and compliance standards.
🔄 Workflow
Campaign Request → AI drafts content → Validator checks tone/compliance → Test deployment simulation → Iteration loop → Final approved campaign assets
🧹 Conflict Resolution
- Resolve conflicting brand guidelines (e.g., tone mismatch between product lines).
- Standardize formatting across channels (social, email, ads).
- Ensure package/library consistency for creative assets (e.g., Canva API vs. Adobe API).

In [25]:
# =========================================
# 🔧 AUTO INSTALL (SAFE)
# =========================================
import sys, subprocess

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import autogen
except:
    install("pyautogen==0.2.25")

try:
    import openai
except:
    install("openai==1.3.8")

# Google Gemini SDK
try:
    import google.generativeai as genai
except:
    install("google-generativeai")

# =========================================
# 🔑 LOAD GOOGLE API KEY (COLAB SECRET)
# =========================================
from google.colab import userdata
import os

key = userdata.get("GEMINI_API_KEY")

if not key:
    raise ValueError("❌ Missing 'google_api_key' in Colab secrets")

os.environ["GOOGLE_API_KEY"] = key

print("✅ Google API Key Loaded")

# =========================================
# 🤖 GEMINI WRAPPER (IMPORTANT)
# =========================================
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

def gemini_chat(prompt):
    model = genai.GenerativeModel("gemini-2.5-flash")
    response = model.generate_content(prompt)
    return response.text

# =========================================
# 🧠 CUSTOM AGENT (NO AUTOGEN LLM)
# =========================================
class GeminiAgent:
    def __init__(self, name, system_message):
        self.name = name
        self.system_message = system_message

    def reply(self, message):
        full_prompt = f"{self.system_message}\n\nUser Task:\n{message}"
        return gemini_chat(full_prompt)

# =========================================
# ✍️ CONTENT CREATOR
# =========================================
creator = GeminiAgent(
    "Content_Creator",
    """
You are a senior marketing content creator.
Create engaging Instagram posts.
Use enthusiastic, eco-friendly, modern tone.
Include CTA and hashtags.
Output in markdown.
"""
)

# =========================================
# 🔎 VALIDATOR
# =========================================
validator = GeminiAgent(
    "Validator",
    """
You are a strict marketing validator.

Check:
- Tone
- Clarity
- CTA
- Brand alignment

If perfect → say ONLY:
FINAL APPROVED CONTENT

Else → give feedback.
"""
)

# =========================================
# 🧪 SIMULATOR
# =========================================
simulator = GeminiAgent(
    "Simulator",
    """
Adapt content for:
1. Instagram
2. Email
3. Ads

Highlight improvements if needed.
"""
)

# =========================================
# 🔁 LOOP CONTROLLER
# =========================================
task = """
EcoSmart Water Bottle launch

Goal: Awareness + Pre-orders
Audience: Eco-conscious + fitness
Features: Sustainable, insulated, sleek
CTA: Pre-order now
Tone: Enthusiastic, eco-friendly, modern
"""

current = task

for i in range(5):
    print(f"\n🔁 Iteration {i+1}\n")

    content = creator.reply(current)
    print("✍️ CONTENT:\n", content)

    review = validator.reply(content)
    print("\n🔎 REVIEW:\n", review)

    if "FINAL APPROVED CONTENT" in review:
        print("\n✅ APPROVED!")
        break

    simulation = simulator.reply(content)
    print("\n🧪 SIMULATION:\n", simulation)

    current = review + "\nImprove content based on this."

print("\n🎯 FINAL OUTPUT:\n", content)

✅ Google API Key Loaded

🔁 Iteration 1

✍️ CONTENT:
 Here are three engaging Instagram posts for the EcoSmart Water Bottle launch, tailored to an eco-conscious and fitness-focused audience, with an enthusiastic, eco-friendly, and modern tone.

---

### **Instagram Post 1: The Grand Introduction**

**(Image: A dynamic shot of the EcoSmart Water Bottle, possibly in a gym setting or an urban outdoor environment, catching the light. Emphasize its sleek design.)**

✨ **Get Ready to Hydrate Smarter!** ✨

The future of hydration just dropped, and it's absolutely brilliant! 🚀 Introducing the **EcoSmart Water Bottle** – your new must-have companion that perfectly blends cutting-edge performance with unwavering planet-first principles.

We're beyond thrilled to bring you a bottle that isn't just sustainable, but also engineered for your active life. Think premium insulation that keeps your drink *perfectly* chilled for 24 hours (or hot for 12!), wrapped in a design so sleek, it'll turn heads eve

# Scenario: AI-Powered Dev Workflow (Corporate Engineering Team)
# 🎯 Problem Statement
# In companies:
# - Developers manually write and test code.
# - Reviewers check for bugs and compliance.
# - Testing and validation are slow and error-prone.
# 👉 Solution
# A multi-agent system powered by Groq API where:
# - 👨‍💻 Coder Agent writes Python code for requested features.
# - 🔍 Reviewer Agent validates logic and ensures best practices.
# - 🧪 Tester Agent generates unit tests and executes them automatically.
# - 🔁 The loop continues until the code passes all tests and meets requirements.
# 🔄 Workflow
# Feature Request → Product Manager breaks into tasks → Coder writes code → Tester validates with unit tests → Reviewer checks compliance → Iteration loop → Final approved codebase
# 🧹 Conflict Resolution
# - Resolve conflicting dependencies (e.g., package version mismatches).
# - Standardize coding style across agents (PEP8 compliance).
# - Ensure Groq API integration avoids latency or model mismatch issues.

In [36]:
# =========================================
# 🔧 INSTALL
# =========================================
import sys, subprocess

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("google-generativeai")

# =========================================
# 🔑 GEMINI SETUP
# =========================================
from google.colab import userdata
import google.generativeai as genai

key = userdata.get("GEMINI_API_KEY")

if not key:
    raise ValueError("❌ Missing google_api_key")

genai.configure(api_key=key)

# =========================================
# 🤖 MODEL CONFIG (BALANCED OUTPUT)
# =========================================
MODEL_NAME = "gemini-2.5-flash-lite"

def gemini_call(system, user):
    model = genai.GenerativeModel(
        MODEL_NAME,
        generation_config={
            "temperature": 0.4,
            "max_output_tokens": 1200,  # ✅ increased for better explanations
        }
    )

    prompt = f"""
{system}

IMPORTANT:
- Provide clear, well-structured responses
- Keep it concise but DO NOT cut important explanations
- Justify decisions where needed

TASK:
{user}
"""

    response = model.generate_content(prompt)
    return response.text.strip()

# =========================================
# 👨‍💻 CODER AGENT
# =========================================
def coder_agent(task):
    return gemini_call(
        """You are a senior Python developer.
Write clean, efficient, well-structured Python code.
Include comments where helpful.""",
        task
    )

# =========================================
# 🧪 TESTER AGENT
# =========================================
def tester_agent(code):
    return gemini_call(
        """You are a QA engineer.
Write unit tests (pytest style), test edge cases, and explain results.""",
        f"Code:\n{code}"
    )

# =========================================
# 🔍 REVIEWER AGENT
# =========================================
def reviewer_agent(code):
    return gemini_call(
        """You are a strict code reviewer.

Check:
- Correctness
- Edge cases
- Code quality (PEP8)
- Efficiency

If perfect → say: FINAL APPROVED CODE
Else → explain issues clearly and suggest fixes.""",
        f"Review this code:\n{code}"
    )

# =========================================
# 🔁 WORKFLOW LOOP
# =========================================
task = """
Build a Python function:
- Input: list of integers
- Output: top 3 most frequent elements
- Tie → smaller number first
- Handle edge cases
"""

current_task = task
final_code = None

for i in range(5):
    print(f"\n🔁 Iteration {i+1}\n")

    # 👨‍💻 CODE
    code = coder_agent(current_task)
    print("👨‍💻 CODE:\n", code)

    # 🧪 TEST
    test_result = tester_agent(code)
    print("\n🧪 TEST:\n", test_result)

    # 🔍 REVIEW
    review = reviewer_agent(code)
    print("\n🔍 REVIEW:\n", review)

    if "FINAL APPROVED CODE" in review:
        final_code = code
        print("\n✅ APPROVED!")
        break

    # feedback loop
    current_task = review + "\nImprove the code accordingly."

# =========================================
# 🎯 FINAL OUTPUT
# =========================================
print("\n🎯 FINAL CODE:\n", final_code)


🔁 Iteration 1

👨‍💻 CODE:
 ```python
from collections import Counter

def top_three_frequent(nums: list[int]) -> list[int]:
    """
    Finds the top 3 most frequent elements in a list of integers.

    In case of a tie in frequency, the smaller number is prioritized.
    Handles edge cases such as empty lists or lists with fewer than 3 unique elements.

    Args:
        nums: A list of integers.

    Returns:
        A list containing the top 3 most frequent elements, sorted by frequency
        (descending) and then by value (ascending) in case of ties.
        Returns an empty list if the input list is empty.
        Returns all unique elements if there are fewer than 3 unique elements.
    """

    # Handle the edge case of an empty input list.
    if not nums:
        return []

    # Use collections.Counter to efficiently count the frequency of each element.
    # Counter is a subclass of dict that is specifically designed for counting hashable objects.
    # It's generally more

Scenario: AI-Powered HR Onboarding System (Corporate HR Team)
🎯 Problem Statement
In companies:
- HR teams manually prepare onboarding documents and checklists.
- Managers validate compliance with policies.
- Employees often face delays in receiving resources and training schedules.
👉 Solution
A multi-agent system where:
- 📝 HR Agent prepares onboarding documents, training schedules, and resource lists.
- ✅ Compliance Agent reviews documents for policy adherence and legal requirements.
- 📂 Resource Agent allocates accounts, devices, and access permissions.
- 🔁 The loop continues until all onboarding steps are validated and complete.
🔄 Workflow
Onboarding Request → HR Agent drafts documents → Compliance Agent validates → Resource Agent provisions accounts/devices → Iteration loop → Final approved onboarding package
🧹 Conflict Resolution
- Resolve conflicting policy requirements (e.g., different regional compliance rules).
- Standardize onboarding templates across departments.
- Ensure system avoids duplicate resource allocation (e.g., multiple accounts for same employee).

In [45]:
import os
from groq import Groq

# Initialize Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# ==========================================
# ● Onboarding Request
# ==========================================
employee_request = {
    "name": "Ankit Raj",
    "role": "Software Engineer",
    "department": "Engineering",
    "location": "India",
    "joining_date": "2026-04-01"
}


# ==========================================
# 📝 HR AGENT
# ==========================================
class HRAgent:

    def create_onboarding_plan(self, employee):
        print("📝 HR Agent: Creating onboarding documents...\n")

        prompt = f"""
        Create a complete onboarding package for:

        Name: {employee['name']}
        Role: {employee['role']}
        Department: {employee['department']}
        Location: {employee['location']}
        Joining Date: {employee['joining_date']}

        Include:
        - Welcome email
        - Onboarding checklist
        - Training schedule
        - Resource requirements

        Ensure standardized format.
        """

        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[{"role": "user", "content": prompt}],
            temperature=1,
            max_tokens=1000
        )

        return response.choices[0].message.content


# ==========================================
# ✅ COMPLIANCE AGENT
# ==========================================
class ComplianceAgent:

    def validate(self, content, employee):
        print("✅ Compliance Agent: Validating policies...\n")

        prompt = f"""
        Validate this onboarding package:

        {content}

        Check:
        - Legal compliance for {employee['location']}
        - HR policy adherence
        - No conflicting rules

        Resolve:
        - Regional conflicts
        - Missing compliance items

        Reply ONLY:
        APPROVED or REJECTED + reason
        """

        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[{"role": "user", "content": prompt}],
            temperature=1,
            max_tokens=1000
        )

        result = response.choices[0].message.content

        if "APPROVED" in result:
            return True, result
        return False, result


# ==========================================
# 📂 RESOURCE AGENT
# ==========================================
class ResourceAgent:

    allocated_resources = set()

    def provision(self, employee):
        print("📂 Resource Agent: Allocating resources...\n")

        resources = [
            f"{employee['name']}_email",
            f"{employee['name']}_laptop",
            f"{employee['name']}_github_access"
        ]

        # Prevent duplicates
        allocated = []
        for r in resources:
            if r not in self.allocated_resources:
                self.allocated_resources.add(r)
                allocated.append(r)

        return allocated


# ==========================================
# 🔄 MAIN WORKFLOW
# ==========================================
def run_hr_onboarding():

    hr = HRAgent()
    compliance = ComplianceAgent()
    resource = ResourceAgent()

    while True:
        # Step 1: HR creates onboarding plan
        plan = hr.create_onboarding_plan(employee_request)

        # Step 2: Compliance validation
        approved, feedback = compliance.validate(plan, employee_request)
        print("Compliance Result:\n", feedback, "\n")

        if approved:
            break
        else:
            print("🔄 Fixing compliance issues...\n")

    # Step 3: Resource provisioning
    resources = resource.provision(employee_request)

    # Final Output
    print("\n✅ FINAL ONBOARDING PACKAGE\n")
    print(plan)

    print("\n📂 RESOURCES ALLOCATED\n")
    print(resources)


# ==========================================
# ▶️ RUN
# ==========================================
run_hr_onboarding()


📝 HR Agent: Creating onboarding documents...



AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

# Task
Modify the `HRAgent` and `ComplianceAgent` classes in cell `-XDLnmmQr5Qd` by setting the `model` to `openai/gpt-oss-20b`, `temperature` to `1`, and adding `max_tokens=1000` to their `client.chat.completions.create` calls. Afterwards, run the updated cell to confirm the HR onboarding system successfully executes and the `AuthenticationError` is resolved.

## Update Groq Client Configuration

### Subtask:
Modify the `HRAgent` and `ComplianceAgent` classes in cell `-XDLnmmQr5Qd` to use the specified model `openai/gpt-oss-20b` and adjust the `temperature` to `1` and `max_tokens` to `1000` in their `client.chat.completions.create` calls. This aligns with the provided `llm_config`.


In [47]:
import os
import requests

# ==========================================
# 🔐 LLM CONFIG (Groq OpenAI-Compatible)
# ==========================================
config_list = [
    {
        "model": "openai/gpt-oss-20b",
        "api_key": os.environ.get("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1",
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 1,
    "max_tokens": 1000,
    "top_p": 1,
}


# ==========================================
# 🔄 GENERIC LLM CALL FUNCTION
# ==========================================
def call_llm(prompt):
    config = llm_config["config_list"][0]

    url = f"{config['base_url']}/chat/completions"

    headers = {
        "Authorization": f"Bearer {config['api_key']}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": config["model"],
        "messages": [{"role": "user", "content": prompt}],
        "temperature": llm_config["temperature"],
        "max_tokens": llm_config["max_tokens"],
        "top_p": llm_config["top_p"],
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        raise Exception(f"API Error: {response.text}")

    return response.json()["choices"][0]["message"]["content"]


# ==========================================
# 👤 Onboarding Request
# ==========================================
employee_request = {
    "name": "Ankit Raj",
    "role": "Software Engineer",
    "department": "Engineering",
    "location": "India",
    "joining_date": "2026-04-01"
}


# ==========================================
# 📝 HR AGENT
# ==========================================
class HRAgent:

    def create_onboarding_plan(self, employee):
        print("📝 HR Agent: Creating onboarding documents...\n")

        prompt = f"""
        Create a complete onboarding package for:

        Name: {employee['name']}
        Role: {employee['role']}
        Department: {employee['department']}
        Location: {employee['location']}
        Joining Date: {employee['joining_date']}

        Include:
        - Welcome email
        - Onboarding checklist
        - Training schedule
        - Resource requirements

        Ensure standardized format.
        """

        return call_llm(prompt)


# ==========================================
# ✅ COMPLIANCE AGENT
# ==========================================
class ComplianceAgent:

    def validate(self, content, employee):
        print("✅ Compliance Agent: Validating policies...\n")

        prompt = f"""
        Validate this onboarding package:

        {content}

        Check:
        - Legal compliance for {employee['location']}
        - HR policy adherence
        - No conflicting rules

        Resolve:
        - Regional conflicts
        - Missing compliance items

        Reply ONLY:
        APPROVED or REJECTED + reason
        """

        result = call_llm(prompt)

        if "APPROVED" in result.upper():
            return True, result
        return False, result


# ==========================================
# 📂 RESOURCE AGENT
# ==========================================
class ResourceAgent:

    def __init__(self):
        self.allocated_resources = set()

    def provision(self, employee):
        print("📂 Resource Agent: Allocating resources...\n")

        resources = [
            f"{employee['name']}_email",
            f"{employee['name']}_laptop",
            f"{employee['name']}_github_access"
        ]

        allocated = []
        for r in resources:
            if r not in self.allocated_resources:
                self.allocated_resources.add(r)
                allocated.append(r)

        return allocated


# ==========================================
# 🔄 MAIN WORKFLOW
# ==========================================
def run_hr_onboarding():

    hr = HRAgent()
    compliance = ComplianceAgent()
    resource = ResourceAgent()

    iteration = 0

    while True:
        iteration += 1
        print(f"\n🔁 Iteration {iteration}\n")

        # Step 1: HR creates onboarding plan
        plan = hr.create_onboarding_plan(employee_request)

        # Step 2: Compliance validation
        approved, feedback = compliance.validate(plan, employee_request)
        print("Compliance Result:\n", feedback, "\n")

        if approved:
            break
        else:
            print("🔄 Fixing compliance issues...\n")

    # Step 3: Resource provisioning
    resources = resource.provision(employee_request)

    # Final Output
    print("\n✅ FINAL ONBOARDING PACKAGE\n")
    print(plan)

    print("\n📂 RESOURCES ALLOCATED\n")
    print(resources)


# ==========================================
# ▶️ RUN
# ==========================================
run_hr_onboarding()


🔁 Iteration 1

📝 HR Agent: Creating onboarding documents...



Exception: API Error: {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}


In [49]:
import os
import google.generativeai as genai

# ==========================================
# 🔐 CONFIG (Gemini)
# ==========================================
genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))

model = genai.GenerativeModel("gemini-2.5-flash")

llm_config = {
    "temperature": 1,
    "max_tokens": 1000,
    "top_p": 1,
}


# ==========================================
# 🔄 GENERIC LLM CALL FUNCTION
# ==========================================
def call_llm(prompt):
    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": llm_config["temperature"],
            "max_output_tokens": llm_config["max_tokens"],
            "top_p": llm_config["top_p"],
        }
    )
    return response.text


# ==========================================
# 👤 Onboarding Request
# ==========================================
employee_request = {
    "name": "Ankit Raj",
    "role": "Software Engineer",
    "department": "Engineering",
    "location": "India",
    "joining_date": "2026-04-01"
}


# ==========================================
# 📝 HR AGENT
# ==========================================
class HRAgent:

    def create_onboarding_plan(self, employee):
        print("📝 HR Agent: Creating onboarding documents...\n")

        prompt = f"""
        Create a complete onboarding package for:

        Name: {employee['name']}
        Role: {employee['role']}
        Department: {employee['department']}
        Location: {employee['location']}
        Joining Date: {employee['joining_date']}

        Include:
        - Welcome email
        - Onboarding checklist
        - Training schedule
        - Resource requirements

        Ensure standardized format.
        """

        return call_llm(prompt)


# ==========================================
# ✅ COMPLIANCE AGENT
# ==========================================
class ComplianceAgent:

    def validate(self, content, employee):
        print("✅ Compliance Agent: Validating policies...\n")

        prompt = f"""
        Validate this onboarding package:

        {content}

        Check:
        - Legal compliance for {employee['location']}
        - HR policy adherence
        - No conflicting rules

        Resolve:
        - Regional conflicts
        - Missing compliance items

        Reply ONLY:
        APPROVED or REJECTED + reason
        """

        result = call_llm(prompt)

        if "APPROVED" in result.upper():
            return True, result
        return False, result


# ==========================================
# 📂 RESOURCE AGENT
# ==========================================
class ResourceAgent:

    def __init__(self):
        self.allocated_resources = set()

    def provision(self, employee):
        print("📂 Resource Agent: Allocating resources...\n")

        resources = [
            f"{employee['name']}_email",
            f"{employee['name']}_laptop",
            f"{employee['name']}_github_access"
        ]

        allocated = []
        for r in resources:
            if r not in self.allocated_resources:
                self.allocated_resources.add(r)
                allocated.append(r)

        return allocated


# ==========================================
# 🔄 MAIN WORKFLOW
# ==========================================
def run_hr_onboarding():

    hr = HRAgent()
    compliance = ComplianceAgent()
    resource = ResourceAgent()

    iteration = 0

    while True:
        iteration += 1
        print(f"\n🔁 Iteration {iteration}\n")

        # Step 1: HR creates onboarding plan
        plan = hr.create_onboarding_plan(employee_request)

        # Step 2: Compliance validation
        approved, feedback = compliance.validate(plan, employee_request)
        print("Compliance Result:\n", feedback, "\n")

        if approved:
            break
        else:
            print("🔄 Fixing compliance issues...\n")

    # Step 3: Resource provisioning
    resources = resource.provision(employee_request)

    # Final Output
    print("\n✅ FINAL ONBOARDING PACKAGE\n")
    print(plan)

    print("\n📂 RESOURCES ALLOCATED\n")
    print(resources)


# ==========================================
# ▶️ RUN
# ==========================================
run_hr_onboarding()


🔁 Iteration 1

📝 HR Agent: Creating onboarding documents...

✅ Compliance Agent: Validating policies...

Compliance Result:
 REJECTED + The onboarding package provided is empty. It only contains a title and a placeholder for the name ("**Name:**"). There is no content to validate against legal compliance for India 

🔄 Fixing compliance issues...


🔁 Iteration 2

📝 HR Agent: Creating onboarding documents...

✅ Compliance Agent: Validating policies...

Compliance Result:
 REJECTED.

**Reason:** The onboarding package for Ankit Raj is incomplete for validation. No content for the package has been provided, preventing any assessment of legal compliance, HR policy adherence 

🔄 Fixing compliance issues...


🔁 Iteration 3

📝 HR Agent: Creating onboarding documents...

✅ Compliance Agent: Validating policies...

Compliance Result:
 REJECTED + The provided onboarding package content is incomplete. It only shows the beginning ("**Employee Name:** An"). To validate for legal compliance, HR poli

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 54.797816584s.

In [51]:
import os
import time
import google.generativeai as genai

# ==========================================
# 🔐 CONFIG
# ==========================================
genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))

model = genai.GenerativeModel("gemini-2.5-flash")

llm_config = {
    "temperature": 0.8,
    "max_tokens": 800,
    "top_p": 1,
}

# ==========================================
# 🔄 SAFE LLM CALL (WITH RETRY)
# ==========================================
def call_llm(prompt, retries=3, delay=60):
    for attempt in range(retries):
        try:
            response = model.generate_content(
                prompt,
                generation_config={
                    "temperature": llm_config["temperature"],
                    "max_output_tokens": llm_config["max_tokens"],
                    "top_p": llm_config["top_p"],
                }
            )
            return response.text

        except Exception as e:
            print(f"⚠️ API Error: {e}")

            if attempt < retries - 1:
                print(f"⏳ Waiting {delay} seconds before retry...\n")
                time.sleep(delay)
            else:
                raise e


# ==========================================
# 👤 Onboarding Request
# ==========================================
employee_request = {
    "name": "Ankit Raj",
    "role": "Software Engineer",
    "department": "Engineering",
    "location": "India",
    "joining_date": "2026-04-01"
}


# ==========================================
# 📝 HR AGENT
# ==========================================
class HRAgent:
    def create_onboarding_plan(self, employee):
        print("📝 HR Agent: Creating onboarding documents...\n")

        prompt = f"""
        Create a complete onboarding package for:

        Name: {employee['name']}
        Role: {employee['role']}
        Department: {employee['department']}
        Location: {employee['location']}
        Joining Date: {employee['joining_date']}

        Include:
        - Welcome email
        - Onboarding checklist
        - Training schedule
        - Resource requirements
        """

        return call_llm(prompt)


# ==========================================
# ✅ COMPLIANCE AGENT
# ==========================================
class ComplianceAgent:
    def validate(self, content, employee):
        print("✅ Compliance Agent: Validating...\n")

        prompt = f"""
        Validate this onboarding package:

        {content}

        Reply ONLY:
        APPROVED or REJECTED + reason
        """

        result = call_llm(prompt)

        if "APPROVED" in result.upper():
            return True, result
        return False, result


# ==========================================
# 📂 RESOURCE AGENT
# ==========================================
class ResourceAgent:
    def __init__(self):
        self.allocated_resources = set()

    def provision(self, employee):
        print("📂 Allocating resources...\n")

        resources = [
            f"{employee['name']}_email",
            f"{employee['name']}_laptop",
            f"{employee['name']}_github"
        ]

        allocated = []
        for r in resources:
            if r not in self.allocated_resources:
                self.allocated_resources.add(r)
                allocated.append(r)

        return allocated


# ==========================================
# 🔄 MAIN WORKFLOW (OPTIMIZED)
# ==========================================
def run_hr_onboarding():

    hr = HRAgent()
    compliance = ComplianceAgent()
    resource = ResourceAgent()

    # 🔥 Only ONE iteration to avoid quota issue
    plan = hr.create_onboarding_plan(employee_request)

    approved, feedback = compliance.validate(plan, employee_request)
    print("Compliance Result:\n", feedback, "\n")

    # If rejected → regenerate ONCE (not infinite loop)
    if not approved:
        print("🔁 Regenerating once...\n")
        plan = hr.create_onboarding_plan(employee_request)

    resources = resource.provision(employee_request)

    print("\n✅ FINAL ONBOARDING PACKAGE\n")
    print(plan)

    print("\n📂 RESOURCES ALLOCATED\n")
    print(resources)


# ==========================================
# ▶️ RUN
# ==========================================
run_hr_onboarding()

📝 HR Agent: Creating onboarding documents...

✅ Compliance Agent: Validating...

Compliance Result:
 REJECTED + The onboarding package content itself has not been provided for validation. Only an introductory welcome message was given. 

🔁 Regenerating once...

📝 HR Agent: Creating onboarding documents...

📂 Allocating resources...


✅ FINAL ONBOARDING PACKAGE

Here's a comprehensive onboarding package tailored for Ankit Raj, a Software Engineer joining the Engineering Department in India on April 1, 20

📂 RESOURCES ALLOCATED

['Ankit Raj_email', 'Ankit Raj_laptop', 'Ankit Raj_github']


🧠 AutoGen Assessment Scenario
🎯 Title: Multi-Agent AI System for Automated Code Review & Execution
🔹 Scenario Overview

You are part of an AI engineering team building an AutoGen-based developer assistant.

The system should:

Accept a user problem statement
Generate code
Review the code
Execute it
Provide feedback and corrections

This must be implemented using multiple collaborating agents.

🔹 Problem Statement

A user gives the following input:

“Write a Python function to find the second largest number in a list and handle edge cases.”

Your AutoGen system should:

Generate correct code
Validate logic
Execute the code
Handle errors
Provide final response
🔹 Requirements
✅ 1. Agent Design (MANDATORY)

Create at least 3 agents:

🔸 Coder Agent
Generates Python code
Should follow best practices
Must handle edge cases
🔸 Reviewer Agent
Reviews code for:
Bugs
Logic errors
Edge cases
Suggests improvements
🔸 Executor Agent (UserProxy)
Runs the code
Returns output/errors
🔹 2. Agent Collaboration Flow

Design the flow such that:

User → Coder → Reviewer → Executor → Reviewer (if error) → Final Output

👉 The system should not stop after one step
👉 It must iterate until correct output is achieved

🔹 3. Functional Expectations

Your system must:

✔ Handle incorrect code generated initially
✔ Ensure reviewer gives structured feedback
✔ Prevent infinite loops
✔ Show final clean output

🔹 4. Technical Constraints
Use AutoGen framework
Use LLM-based agents
Enable code execution
Add termination condition
🔹 5. Edge Cases to Handle

Test your system with:

Empty list
List with one element
Duplicate values
Negative numbers

In [5]:
import os
from groq import Groq

# Initialize Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# ==========================================
# 👨‍💻 CODER AGENT
# ==========================================
class CoderAgent:
    def generate_code(self, task):
        print("👨‍💻 Coder Agent: Generating code...")

        prompt = f"""
        You are a senior Python developer.
        Write a Python function to find the second largest number in a list and handle edge cases.

        Requirements:
        - The function should be named `find_second_largest`.
        - Input: A list of numbers.
        - Output: The second largest number.
        - Handle edge cases: empty list, list with one element, list with fewer than two unique elements.
        - If no second largest number exists (e.g., list is too short or all elements are the same), return None.
        - Ensure the code is clean, efficient, well-structured, and well-commented.
        - Follow Python best practices (PEP8).

        Example:
        - `[1, 2, 3, 4, 5]` should return `4`
        - `[5, 5, 5]` should return `None`
        - `[1]` should return `None`
        - `[]` should return `None`
        - `[1, 2, 2, 3]` should return `2`
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant", # Using a suitable Groq model
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7, # Allows for creative code generation
            max_tokens=500 # Sufficiently detailed code
        )

        return response.choices[0].message.content


# Example usage (for testing purposes, will be removed later if not needed in final flow)
# coder = CoderAgent()
# generated_code = coder.generate_code("Write a Python function to find the second largest number in a list and handle edge cases.")
# print(generated_code)


In [2]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 9.1 MB/s eta 0:00:00


In [7]:
import os
from groq import Groq

# Initialize Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# ==========================================
# 👨‍💻 CODER AGENT
# ==========================================
class CoderAgent:
    def generate_code(self, task):
        print("👨‍💻 Coder Agent: Generating code...")

        prompt = f"""
        You are a senior Python developer.
        Write a Python function to find the second largest number in a list and handle edge cases.

        Requirements:
        - The function should be named `find_second_largest`.
        - Input: A list of numbers.
        - Output: The second largest number.
        - Handle edge cases: empty list, list with one element, list with fewer than two unique elements.
        - If no second largest number exists (e.g., list is too short or all elements are the same), return None.
        - Ensure the code is clean, efficient, well-structured, and well-commented.
        - Follow Python best practices (PEP8).

        Example:
        - `[1, 2, 3, 4, 5]` should return `4`
        - `[5, 5, 5]` should return `None`
        - `[1]` should return `None`
        - `[]` should return `None`
        - `[1, 2, 2, 3]` should return `2`
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant", # Using a suitable Groq model
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7, # Allows for creative code generation
            max_tokens=500 # Sufficiently detailed code
        )

        return response.choices[0].message.content


# Example usage (for testing purposes, will be removed later if not needed in final flow)
# coder = CoderAgent()
# generated_code = coder.generate_code("Write a Python function to find the second largest number in a list and handle edge cases.")
# print(generated_code)

In [6]:
import os
from groq import Groq
from google.colab import userdata

# Load GROQ_API_KEY from Colab secrets and set as environment variable
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Initialize Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# ==========================================
# 👨‍💻 CODER AGENT
# ==========================================
class CoderAgent:
    def generate_code(self, task):
        print("👨‍💻 Coder Agent: Generating code...")

        prompt = f"""
        You are a senior Python developer.
        Write a Python function to find the second largest number in a list and handle edge cases.

        Requirements:
        - The function should be named `find_second_largest`.
        - Input: A list of numbers.
        - Output: The second largest number.
        - Handle edge cases: empty list, list with one element, list with fewer than two unique elements.
        - If no second largest number exists (e.g., list is too short or all elements are the same), return None.
        - Ensure the code is clean, efficient, well-structured, and well-commented.
        - Follow Python best practices (PEP8).

        Example:
        - `[1, 2, 3, 4, 5]` should return `4`
        - `[5, 5, 5]` should return `None`
        - `[1]` should return `None`
        - `[]` should return `None`
        - `[1, 2, 2, 3]` should return `2`
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant", # Using a suitable Groq model
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7, # Allows for creative code generation
            max_tokens=500 # Sufficiently detailed code
        )

        return response.choices[0].message.content


# Example usage (for testing purposes, will be removed later if not needed in final flow)
# coder = CoderAgent()
# generated_code = coder.generate_code("Write a Python function to find the second largest number in a list and handle edge cases.")
# print(generated_code)


In [8]:
import os
from groq import Groq

# Initialize Groq client (ensure this is done once in the notebook or passed)
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# ==========================================
# 🔍 REVIEWER AGENT
# ==========================================
class ReviewerAgent:
    def review_code(self, code):
        print("🔍 Reviewer Agent: Reviewing code...")

        prompt = f"""
        You are a strict code reviewer. Your task is to thoroughly review the provided Python code.

        Code to review:
        ```python
        {code}
        ```

        Carefully check for the following:
        - **Correctness**: Does the code accurately solve the problem of finding the second largest number?
        - **Logic Errors**: Are there any flaws in the algorithm or implementation?
        - **Edge Case Handling**: Does the code correctly handle:
            - An empty list (`[]`)
            - A list with only one element (e.g., `[5]`)
            - A list with fewer than two unique elements (e.g., `[5, 5, 5]`)
            - Lists with duplicate values (e.g., `[1, 2, 2, 3]`)
            - Lists with negative numbers (e.g., `[-1, -5, -2]`)
        - **Code Quality**: Adherence to PEP8 guidelines, readability, clarity of variable names, and appropriate commenting.
        - **Efficiency**: Is the solution reasonably efficient for typical input sizes?

        If the code is absolutely perfect and meets all requirements, including handling all specified edge cases correctly and adhering to best practices, reply ONLY with the phrase: 'FINAL APPROVED CODE'.

        Otherwise, provide clear, concise, and actionable feedback. Explain any issues found (bugs, logical errors, edge case failures, style violations, inefficiencies) and suggest specific fixes or improvements. Your feedback should guide the developer to correct the code effectively.
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant", # Suitable Groq model for code review
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5, # Balanced creativity and accuracy for feedback
            max_tokens=1000 # Sufficient tokens for detailed feedback
        )

        return response.choices[0].message.content


# Example usage (for testing purposes)
# reviewer = ReviewerAgent()
# sample_code = """def add(a, b): return a + b"""
# review_feedback = reviewer.review_code(sample_code)
# print(review_feedback)

## Define Executor Agent (UserProxy)

### Subtask:
Implement the `executor_agent` (UserProxy equivalent) that can execute the Python code generated. This agent will capture the output and any errors during execution, which will then be fed back to the Reviewer Agent.


**Reasoning**:
I will define the `ExecutorAgent` class with the `execute_code` method to execute Python code, capture output and errors, and handle file cleanup as per the instructions.



In [9]:
import os
import subprocess

# ==========================================
# 🧪 EXECUTOR AGENT (UserProxy Equivalent)
# ==========================================
class ExecutorAgent:
    def execute_code(self, code):
        print("🧪 Executor Agent: Executing code...")
        temp_file_name = "temp_code.py"

        try:
            # Write the code to a temporary file
            with open(temp_file_name, "w") as f:
                f.write(code)

            # Execute the temporary file using subprocess
            process = subprocess.run(
                ["python", temp_file_name],
                capture_output=True,
                text=True, # Decode stdout/stderr as text
                check=False # Do not raise an exception for non-zero exit codes
            )

            stdout = process.stdout
            stderr = process.stderr
            success = process.returncode == 0

            print(f"Stdout:\n{stdout}")
            if stderr:
                print(f"Stderr:\n{stderr}")

            return {"stdout": stdout, "stderr": stderr, "success": success}

        except Exception as e:
            return {"stdout": "", "stderr": str(e), "success": False}

        finally:
            # Clean up the temporary file
            if os.path.exists(temp_file_name):
                os.remove(temp_file_name)

# Example usage (for testing purposes)
# executor = ExecutorAgent()
# sample_code = "print('Hello from Executor')\nraise ValueError('Test error')"
# execution_result = executor.execute_code(sample_code)
# print(execution_result)


In [10]:
import os
from groq import Groq
from google.colab import userdata
import subprocess

# Load GROQ_API_KEY from Colab secrets and set as environment variable
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Initialize Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# ==========================================
# 👨‍💻 CODER AGENT
# ==========================================
class CoderAgent:
    def generate_code(self, task):
        print("👨‍💻 Coder Agent: Generating code...")

        prompt = f"""
        You are a senior Python developer.
        Write a Python function to find the second largest number in a list and handle edge cases.

        Requirements:
        - The function should be named `find_second_largest`.
        - Input: A list of numbers.
        - Output: The second largest number.
        - Handle edge cases: empty list, list with one element, list with fewer than two unique elements.
        - If no second largest number exists (e.g., list is too short or all elements are the same), return None.
        - Ensure the code is clean, efficient, well-structured, and well-commented.
        - Follow Python best practices (PEP8).

        Example:
        - `[1, 2, 3, 4, 5]` should return `4`
        - `[5, 5, 5]` should return `None`
        - `[1]` should return `None`
        - `[]` should return `None`
        - `[1, 2, 2, 3]` should return `2`

        Reviewer feedback for improvement: {task}
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant", # Using a suitable Groq model
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7, # Allows for creative code generation
            max_tokens=500 # Sufficiently detailed code
        )

        return response.choices[0].message.content


# ==========================================
# 🔍 REVIEWER AGENT
# ==========================================
class ReviewerAgent:
    def review_code(self, code, execution_result=None):
        print("🔍 Reviewer Agent: Reviewing code...")

        feedback_message = ""
        if execution_result:
            feedback_message = f"\nExecution Result:\nStdout:\n{execution_result['stdout']}\nStderr:\n{execution_result['stderr']}\nSuccess: {execution_result['success']}"
            if not execution_result['success']:
                feedback_message += "\n\nBased on the execution results, the code failed. Please fix the errors.\n"
            else:
                feedback_message += "\n\nBased on the execution results, the code ran successfully. Please ensure the output is correct according to the problem statement.\n"

        prompt = f"""
        You are a strict code reviewer. Your task is to thoroughly review the provided Python code.

        Code to review:
        ```python
        {code}
        ```

        {feedback_message}

        Carefully check for the following:
        - **Correctness**: Does the code accurately solve the problem of finding the second largest number?
        - **Logic Errors**: Are there any flaws in the algorithm or implementation?
        - **Edge Case Handling**: Does the code correctly handle:
            - An empty list (`[]`)
            - A list with only one element (e.g., `[5]`)
            - A list with fewer than two unique elements (e.g., `[5, 5, 5]`)
            - Lists with duplicate values (e.g., `[1, 2, 2, 3]`)
            - Lists with negative numbers (e.g., `[-1, -5, -2]`)
        - **Code Quality**: Adherence to PEP8 guidelines, readability, clarity of variable names, and appropriate commenting.
        - **Efficiency**: Is the solution reasonably efficient for typical input sizes?

        If the code is absolutely perfect and meets all requirements, including handling all specified edge cases correctly and adhering to best practices, reply ONLY with the phrase: 'FINAL APPROVED CODE'.

        Otherwise, provide clear, concise, and actionable feedback. Explain any issues found (bugs, logical errors, edge case failures, style violations, inefficiencies) and suggest specific fixes or improvements. Your feedback should guide the developer to correct the code effectively.
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant", # Suitable Groq model for code review
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5, # Balanced creativity and accuracy for feedback
            max_tokens=1000 # Sufficient tokens for detailed feedback
        )

        return response.choices[0].message.content


# ==========================================
# 🧪 EXECUTOR AGENT (UserProxy Equivalent)
# ==========================================
class ExecutorAgent:
    def execute_code(self, code):
        print("🧪 Executor Agent: Executing code...")
        temp_file_name = "temp_code.py"

        try:
            # Write the code to a temporary file
            with open(temp_file_name, "w") as f:
                f.write(code)

            # Execute the temporary file using subprocess
            process = subprocess.run(
                ["python", temp_file_name],
                capture_output=True,
                text=True, # Decode stdout/stderr as text
                check=False # Do not raise an exception for non-zero exit codes
            )

            stdout = process.stdout
            stderr = process.stderr
            success = process.returncode == 0

            print(f"Stdout:\n{stdout}")
            if stderr:
                print(f"Stderr:\n{stderr}")

            return {"stdout": stdout, "stderr": stderr, "success": success}

        except Exception as e:
            return {"stdout": "", "stderr": str(e), "success": False}

        finally:
            # Clean up the temporary file
            if os.path.exists(temp_file_name):
                os.remove(temp_file_name)



# ==========================================
# 🔄 MAIN WORKFLOW
# ==========================================
def run_code_review_system(user_problem):
    coder = CoderAgent()
    reviewer = ReviewerAgent()
    executor = ExecutorAgent()

    current_task = user_problem
    approved_code = None
    iteration = 0
    max_iterations = 5 # Prevent infinite loops

    print("Starting Multi-Agent AI System for Automated Code Review & Execution")
    print(f"Initial Problem: {user_problem}")

    while approved_code is None and iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")

        # 1. Coder generates code
        generated_code = coder.generate_code(current_task)
        print("\nGenerated Code:\n```python")
        print(generated_code)
        print("```")

        # 2. Executor executes the code
        # To properly test the generated function, we need to add test cases
        # to the code before execution.
        test_cases = """

# --- Test Cases for find_second_largest function ---
print(f"[1, 2, 3, 4, 5] -> {find_second_largest([1, 2, 3, 4, 5])}") # Expected: 4
print(f"[5, 5, 5] -> {find_second_largest([5, 5, 5])}")           # Expected: None
print(f"[1] -> {find_second_largest([1])}")                      # Expected: None
print(f"[] -> {find_second_largest([])}")                         # Expected: None
print(f"[1, 2, 2, 3] -> {find_second_largest([1, 2, 2, 3])}")      # Expected: 2
print(f"[-1, -5, -2] -> {find_second_largest([-1, -5, -2])}")    # Expected: -2
print(f"[10, 20, 30] -> {find_second_largest([10, 20, 30])}")    # Expected: 20
print(f"[7, 7, 7, 8, 8] -> {find_second_largest([7, 7, 7, 8, 8])}") # Expected: 7
"""
        code_to_execute = generated_code + test_cases
        execution_result = executor.execute_code(code_to_execute)

        # 3. Reviewer reviews the code and execution results
        review_feedback = reviewer.review_code(generated_code, execution_result)
        print("\nReviewer Feedback:\n", review_feedback)

        if "FINAL APPROVED CODE" in review_feedback.upper():
            approved_code = generated_code
            print("\n✅ FINAL APPROVED CODE")
        else:
            # Update task for next iteration with reviewer feedback
            current_task = f"Based on the following feedback, refine the previous code:\n{review_feedback}"
            if iteration == max_iterations -1:
                print("Max iterations almost reached. One more try.")

    if approved_code:
        print("\n🎯 Final Code:\n```python")
        print(approved_code)
        print("```")
    else:
        print("\n❌ Could not get approved code within maximum iterations.")


# --- RUN THE SYSTEM ---
user_problem_statement = "Write a Python function to find the second largest number in a list and handle edge cases."
run_code_review_system(user_problem_statement)


Starting Multi-Agent AI System for Automated Code Review & Execution
Initial Problem: Write a Python function to find the second largest number in a list and handle edge cases.

--- Iteration 1 ---
👨‍💻 Coder Agent: Generating code...

Generated Code:
```python
```python
def find_second_largest(numbers):
    """
    Finds the second largest number in a list.

    Args:
        numbers (list): A list of numbers.

    Returns:
        The second largest number if it exists, otherwise None.
    """

    # Check if the list is empty
    if not numbers:
        return None

    # Remove duplicates by converting the list to a set
    unique_numbers = set(numbers)

    # If the list has fewer than two unique elements, return None
    if len(unique_numbers) < 2:
        return None

    # Remove the largest number from the set
    unique_numbers.remove(max(unique_numbers))

    # Return the largest number from the updated set
    return max(unique_numbers)
```

Example use cases:
```python
prin

In [11]:
user_problem_statement = "Write a Python function to find the second largest number in a list and handle edge cases."
run_code_review_system(user_problem_statement)

Starting Multi-Agent AI System for Automated Code Review & Execution
Initial Problem: Write a Python function to find the second largest number in a list and handle edge cases.

--- Iteration 1 ---
👨‍💻 Coder Agent: Generating code...

Generated Code:
```python
Here's a Python function that meets all the requirements and edge cases:
```python
def find_second_largest(numbers):
    """
    Find the second largest number in a list.

    Args:
        numbers (list): A list of numbers.

    Returns:
        int or None: The second largest number if it exists, otherwise None.
    """
    # Check if the input is a list
    if not isinstance(numbers, list):
        raise ValueError("Input must be a list")

    # Check if the list is empty
    if len(numbers) == 0:
        return None

    # Remove duplicates by converting the list to a set
    unique_numbers = set(numbers)

    # Check if the list has less than two unique elements
    if len(unique_numbers) < 2:
        return None

    # Sort